# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, columns and their `@id`s.

This step discovers what entities are available to be extracted, referencing everything by its `@id`.

In [ ]:
# Explore available record sets and their fields
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f"  Record Set: {rs.id} (name: {getattr(rs, 'name', 'N/A')})")
        print("    Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"      Field: {field.id} (name: {getattr(field, 'name', 'N/A')}, type: {getattr(field, 'data_type', 'N/A')})")
        print("    Columns:")
        for col in getattr(rs, 'columns', []):
            print(f"      Column: {col.id} (name: {getattr(col, 'name', 'N/A')})")


## 3. Data Extraction
Load data from the record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Store IDs for the available record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Preview loaded record sets
print("Parsing record sets with the following IDs:")
for rid in record_set_ids:
    print(f"  {rid}")

# Load each record set as a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame columns for record set '{rs_id}': {list(df.columns)}")

# Preview the first record set if any
if record_set_ids:
    sample_df = dataframes[record_set_ids[0]]
    print(f"\nPreview of record set '{record_set_ids[0]}':")
    display(sample_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and aggregation. All fields and columns are referenced by their `@id`.

In [ ]:
# Identify numeric fields by scanning columns for float/int types
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[rs_id] if rs_id else pd.DataFrame()

numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric columns (@id):", numeric_columns)

# Choose a numeric field for demonstration, e.g. age or another relevant variable
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a categorical field
    group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])] # possible group fields
    group_field_id = group_fields[0] if group_fields else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No numeric columns available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: histogram of numeric field, colored by grouping field if available
if rs_id and numeric_columns:
    numeric_field_id = numeric_columns[0]
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True, ax=ax)
    ax.set_title(f"Distribution of '{numeric_field_id}'")
    ax.set_xlabel(numeric_field_id)
    plt.show()

    # If group_field is available, plot boxplot
    if group_field_id:
        fig, ax = plt.subplots(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], ax=ax)
        ax.set_title(f"'{numeric_field_id}' by '{group_field_id}'")
        ax.set_xlabel(group_field_id)
        ax.set_ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and records with `mlcroissant`.
- Discovered available record sets and their fields via unique `@id`s.
- Extracted tabular data and performed basic filtering, normalization, and grouping based on column `@id`.
- Visualized numeric distribution and categorical differences in the dataset.

This notebook can be extended for further investigation, customized EDA, or machine learning analysis as required.